Librairies

In [27]:
import os
import numpy as np
import matplotlib.pyplot as plt
import operator
import torch
import torch.nn as nn
from PIL import Image
from transformers import ViTFeatureExtractor, ViTModel
from sentence_transformers import SentenceTransformer

Paramètres

In [28]:
MULTIMODAL_FEATURES_DIR = 'MULTIMODAL'
ORIGINAL_IMAGES_DIR = 'MIR_DATASETS_B'
QUERY_IMAGE_KEY = '1_0_chiens_Siberianhusky_901'
QUERY_TEXT = "Husky"
NUM_NEIGHBORS_TO_DISPLAY = 10
TARGET_MODALITY_EMBEDDING_DIM = 384 # Dimension de l'embedding pour SentenceTransformer

Modeles

In [29]:
vit_feature_extractor = ViTFeatureExtractor.from_pretrained('google/vit-base-patch16-224-in21k')
vit_model = ViTModel.from_pretrained('google/vit-base-patch16-224-in21k')
vit_model.eval() # Passer en mode évaluation
class ProjectionLayer(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(ProjectionLayer, self).__init__()
        self.projection = nn.Linear(input_dim, output_dim)

    def forward(self, x):
        return self.projection(x)
vit_input_dim = vit_model.config.hidden_size # Ceci est 768 pour vit-base
vit_projection_layer = ProjectionLayer(vit_input_dim, TARGET_MODALITY_EMBEDDING_DIM)
vit_projection_layer.eval() 
text_model = SentenceTransformer('paraphrase-MiniLM-L6-v2')
text_model.eval() # Passer en mode évaluation


def get_ground_truth(query_key, query_text):
    global features_data
    relevant_keys = set() # Utiliser un set pour éviter les doublons

    keywords_from_image_query = []
    if query_key:
        parts = query_key.lower().split('_')
        if len(parts) > 2: 
            keywords_from_image_query.append(parts[2])
        if len(parts) > 3: 
             keywords_from_image_query.append(parts[3])

    keywords_from_text_query = []
    if query_text:
        # Exemples simples, à affiner pour une meilleure extraction NLP
        query_text_lower = query_text.lower()
        if "spider" in query_text_lower or "araignee" in query_text_lower:
            keywords_from_text_query.append("araignees")
        if "dog" in query_text_lower or "chien" in query_text_lower:
            keywords_from_text_query.append("chiens")
        if "monkey" in query_text_lower or "singe" in query_text_lower:
            keywords_from_text_query.append("singes")
        if "bird" in query_text_lower or "oiseau" in query_text_lower:
            keywords_from_text_query.append("oiseaux")
        if "golden retriever" in query_text_lower:
            keywords_from_text_query.append("chiens")
        if "wolf spider" in query_text_lower:
            keywords_from_text_query.append("araignees")
        if "barnspiderr" in query_text_lower:
            keywords_from_text_query.append("araignees")
        if "bluejay" in query_text_lower:
            keywords_from_text_query.append("oiseau")
        if "baboon" in query_text_lower:
            keywords_from_text_query.append("singe")

    all_query_keywords = list(set(keywords_from_image_query + keywords_from_text_query))

    if not all_query_keywords:
        print(f"Avertissement: Aucun mot-clé pertinent trouvé pour la requête '{query_key}' / '{query_text}'. La vérité terrain sera vide.")
        return []
    for k in features_data.keys():
        k_lower = k.lower()
        for keyword in all_query_keywords:
            if keyword in k_lower:
                relevant_keys.add(k)
                break 

    return list(relevant_keys)

Fonctions utilitaires

In [30]:
def euclidean_distance(vec1, vec2):
    return np.linalg.norm(vec1 - vec2)

def load_features(features_dir):
    """Charge les vecteurs de caractéristiques à partir d'un répertoire spécifié."""
    features_data = {}
    for filename in os.listdir(features_dir):
        if filename.endswith('.txt'): # Supposant des fichiers .txt maintenant
            key = os.path.splitext(filename)[0]
            file_path = os.path.join(features_dir, filename)
            try:
                with open(file_path, 'r') as f:
                    data_str = f.read().strip()
                    # Supposant que les nombres sont séparés par des espaces ; ajustez 'sep' si nécessaire (ex: sep=',')
                    features_data[key] = np.fromstring(data_str, sep=' ')
            except Exception as e:
                print(f"Erreur lors du chargement de {filename}: {e}")
    return features_data



def calculate_precision_recall(results, ground_truth, k_max):
    """Calcule la précision et le rappel à différents rangs."""
    if not ground_truth:
        print("Avertissement : Aucune vérité terrain fournie pour la requête. Impossible de calculer la courbe R-P.")
        return [], []

    precision_values = []
    recall_values = []
    num_relevant_found = 0

    for i in range(1, k_max + 1):
        if i <= len(results):
            retrieved_item_key = results[i - 1][0]
            if retrieved_item_key in ground_truth:
                num_relevant_found += 1

            precision = num_relevant_found / i
            recall = num_relevant_found / len(ground_truth)

            precision_values.append(precision)
            recall_values.append(recall)
        else:
            break # Si moins de résultats que k_max

    return recall_values, precision_values


Calcul Embedding

In [31]:
def get_image_embedding(image_path, projection_layer):
    try:
        image = Image.open(image_path).convert("RGB")
        inputs = vit_feature_extractor(images=image, return_tensors="pt")
        with torch.no_grad():
            outputs = vit_model(**inputs)
            projected_embedding = projection_layer(outputs.pooler_output)
        return projected_embedding.squeeze().numpy()
    except Exception as e:
        print(f"Erreur lors du traitement de l'image {image_path}: {e}")
        return None

def get_text_embedding(text):
    try:
        embedding = text_model.encode(text, convert_to_tensor=True)
        return embedding.cpu().numpy()
    except Exception as e:
        print(f"Erreur lors du traitement du texte : {e}")
        return None

In [32]:
print(f"Chargement des fonctionnalités multimodales depuis {MULTIMODAL_FEATURES_DIR}...")
features_data = load_features(MULTIMODAL_FEATURES_DIR)
print(f"Chargé {len(features_data)} fonctionnalités multimodales.")

if features_data:
    # 2. Générer l'embedding de la requête multimodale
    query_image_path = os.path.join(ORIGINAL_IMAGES_DIR, f"{QUERY_IMAGE_KEY}.jpg")
    query_image_embedding = get_image_embedding(query_image_path, vit_projection_layer)
    query_text_embedding = get_text_embedding(QUERY_TEXT)

    final_query_embedding = None
    final_query_key = f"Image: {QUERY_IMAGE_KEY}, Texte: '{QUERY_TEXT}'"

    if query_image_embedding is not None and query_text_embedding is not None:
        if (query_image_embedding.shape[0] == TARGET_MODALITY_EMBEDDING_DIM and
            query_text_embedding.shape[0] == TARGET_MODALITY_EMBEDDING_DIM):

            final_query_embedding = np.concatenate((query_image_embedding, query_text_embedding))
            print(f"Embedding de requête multimodal généré pour : {final_query_key} avec une dimension totale de {final_query_embedding.shape[0]}.")
        else:
            print(f"Erreur : Incompatibilité de dimension pour les embeddings projetés. Embedding image : {query_image_embedding.shape[0]}, Embedding texte : {query_text_embedding.shape[0]}. Attendu {TARGET_MODALITY_EMBEDDING_DIM} pour les deux.")
            final_query_embedding = None
    else:
        print("Échec de la génération d'un ou des deux embeddings de requête. Sortie.")
        final_query_embedding = None

    if final_query_embedding is not None:
        # 3. Trouver les K plus proches voisins
        print(f"Recherche des {NUM_NEIGHBORS_TO_DISPLAY} voisins les plus proches...")
        distances = {}
        for img_key, features in features_data.items():
            if features.shape == final_query_embedding.shape:
                distances[img_key] = euclidean_distance(final_query_embedding, features)
            else:
                print(f"Avertissement : Incompatibilité de dimension des fonctionnalités pour {img_key}. Attendu {final_query_embedding.shape}, obtenu {features.shape}. Ignoré.")

        sorted_results = sorted(distances.items(), key=operator.itemgetter(1))[:NUM_NEIGHBORS_TO_DISPLAY]

        print("\n--- Voisins les plus proches ---")
        num_cols = 5
        num_rows = NUM_NEIGHBORS_TO_DISPLAY // num_cols
        if NUM_NEIGHBORS_TO_DISPLAY % num_cols != 0:
            num_rows += 1

        fig, axes = plt.subplots(num_rows, num_cols, figsize=(15, 3 * num_rows))
        axes = axes.flatten()

        for i, (img_key, dist) in enumerate(sorted_results):
            try:
                img_path = os.path.join(ORIGINAL_IMAGES_DIR, f"{img_key}.jpg")
                img = Image.open(img_path).convert("RGB")
                axes[i].imshow(img)
                axes[i].set_title(f"Dist: {dist:.2f}\n{img_key}", fontsize=8)
                axes[i].axis('off')
            except Exception as e:
                print(f"Erreur lors de l'affichage de l'image {img_key}: {e}")
                axes[i].text(0.5, 0.5, f"Erreur: {img_key}", ha='center', va='center')
                axes[i].axis('off')

        for j in range(i + 1, num_rows * num_cols):
            axes[j].axis('off')

        plt.tight_layout()
        plt.savefig('nearest_neighbors_results.png')
        plt.close(fig)

        # 4. Calculer et afficher la courbe R-P
        print("\n--- Génération de la courbe Rappel-Précision ---")
        # L'appel à get_ground_truth a été mis à jour ici
        ground_truth_for_query = get_ground_truth(QUERY_IMAGE_KEY, QUERY_TEXT)
        k_rp_max = len(sorted_results)

        if not ground_truth_for_query:
            print("Impossible de générer la courbe R-P : Aucune vérité terrain disponible pour la requête.")
        else:
            recall, precision = calculate_precision_recall(sorted_results, ground_truth_for_query, k_rp_max)

            if recall and precision:
                plt.figure(figsize=(8, 6))
                plt.plot(recall, precision, marker='o', linestyle='-', color='b')
                plt.xlabel('Rappel (Recall)')
                plt.ylabel('Précision (Precision)')
                plt.title(f'Courbe Rappel-Précision pour la requête: {final_query_key}')
                plt.grid(True)
                plt.xlim([0, 1])
                plt.ylim([0, 1])
                plt.savefig('precision_recall_curve.png')
                plt.close()
            else:
                print("Impossible de générer la courbe R-P. Vérifiez les messages d'erreur précédents.")
    else:
        print("Aucune image de requête valide trouvée ou features_data vide. Arrêt du script.")

Chargement des fonctionnalités multimodales depuis MULTIMODAL...
Chargé 2665 fonctionnalités multimodales.
Embedding de requête multimodal généré pour : Image: 1_0_chiens_Siberianhusky_901, Texte: 'Husky' avec une dimension totale de 768.
Recherche des 10 voisins les plus proches...

--- Voisins les plus proches ---

--- Génération de la courbe Rappel-Précision ---
